# 00 — Preflight, data gates, and execution order

Validates the local environment and runs or reviews the non-negotiable metadata/media gates.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def read_optional_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    if stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists():
        return read_stage(relative_without_suffix)
    print("OPTIONAL TABLE NOT AVAILABLE:", relative_without_suffix)
    return pd.DataFrame()

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


Set `RUN_PIPELINE_STAGES=True` only after `config/project.yaml` points to the updated data root. The audit is intentionally run before any signal processing.

In [ ]:
RUN_PIPELINE_STAGES = False

if RUN_PIPELINE_STAGES:
    run_cli("audit")
    run_cli("inventory")
else:
    print("Dry review only. Set RUN_PIPELINE_STAGES=True to run audit and inventory.")


In [ ]:
audit_summary = read_stage("00_audit/bamboo_audit_summary")
cross_workbook = read_stage("00_audit/cross_workbook_summary")
metadata_issues = read_stage("00_audit/bamboo_audit_issues")
inventory = read_stage("00_audit/bamboo_media_inventory")

display(audit_summary)
display(cross_workbook)

issue_counts = (
    metadata_issues.groupby(["severity", "issue"], dropna=False)
    .size().rename("n").reset_index().sort_values(["severity", "n"], ascending=[True, False])
)
save_table(issue_counts, "00_preflight", "metadata_issue_counts")
display(issue_counts)

media_summary = pd.DataFrame({
    "recordings_on_disk": [inventory["file_name"].nunique()],
    "physical_files": [len(inventory)],
    "extensions": [", ".join(sorted(inventory["extension"].dropna().astype(str).unique())) if "extension" in inventory else "not available"],
    "probe_failures": [int(inventory.get("probe_status", pd.Series(dtype=str)).astype(str).ne("ok").sum()) if "probe_status" in inventory else np.nan],
})
save_table(media_summary, "00_preflight", "media_inventory_summary")
display(media_summary)


In [ ]:
# Human-QC folder design can be checked before expensive signal processing.
import yaml

project_cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
schema_path = ROOT / "config" / "human_qc_schema.yaml"
assert schema_path.exists(), "Copy config/human_qc_schema.example.yaml to config/human_qc_schema.yaml."
human_schema = yaml.safe_load(schema_path.read_text(encoding="utf-8"))
data_root = Path(project_cfg["paths"]["data_root"])
human_root = data_root / project_cfg["paths"]["detailed_human_qc"]
ra_names = human_schema["rater_directory_names"]
reliability_root = human_root / human_schema.get("reliability_subdirectory", "Reliability")

main_sets = {
    ra: {path.name for path in (human_root / ra).rglob("*.csv")}
    if (human_root / ra).exists() else set()
    for ra in ra_names
}
reliability_sets = {
    ra: {path.name for path in (reliability_root / ra).rglob("*.csv")}
    if (reliability_root / ra).exists() else set()
    for ra in ra_names
}
folder_counts = pd.DataFrame([
    {
        "rater_id": ra,
        "main_csv_files": len(main_sets[ra]),
        "reliability_csv_files": len(reliability_sets[ra]),
        "main_directory_exists": (human_root / ra).exists(),
        "reliability_directory_exists": (reliability_root / ra).exists(),
    }
    for ra in ra_names
])
save_table(folder_counts, "00_preflight", "human_qc_folder_counts")
display(folder_counts)

main_owners = {}
for ra, names in main_sets.items():
    for name in names:
        main_owners.setdefault(name, []).append(ra)
main_overlap = pd.DataFrame([
    {"export_file": name, "n_main_raters": len(owners), "main_raters": "|".join(owners)}
    for name, owners in sorted(main_owners.items()) if len(owners) > 1
], columns=["export_file", "n_main_raters", "main_raters"])
save_table(main_overlap, "00_preflight", "unexpected_main_assignment_overlap")

reliability_union = set().union(*reliability_sets.values()) if reliability_sets else set()
reliability_intersection = set.intersection(*reliability_sets.values()) if reliability_sets else set()
reliability_gaps = pd.DataFrame([
    {
        "export_file": name,
        "n_raters_present": sum(name in reliability_sets[ra] for ra in ra_names),
        "missing_raters": "|".join(ra for ra in ra_names if name not in reliability_sets[ra]),
    }
    for name in sorted(reliability_union)
    if any(name not in reliability_sets[ra] for ra in ra_names)
], columns=["export_file", "n_raters_present", "missing_raters"])
save_table(reliability_gaps, "00_preflight", "reliability_filename_coverage_gaps")

design_check = pd.DataFrame([{
    "main_files_have_one_assignment_by_export_name": len(main_overlap) == 0,
    "reliability_union_files": len(reliability_union),
    "reliability_files_common_to_all_four_raters": len(reliability_intersection),
    "reliability_files_with_filename_coverage_gaps": len(reliability_gaps),
    "primary_agreement_gate": (
        "provisional_pass"
        if len(reliability_intersection) > 0 and len(reliability_gaps) == 0
        else "review_required"
    ),
}])
save_table(design_check, "00_preflight", "human_qc_folder_design_check")
display(design_check)
if len(main_overlap):
    display(main_overlap.head(50))
if len(reliability_gaps):
    display(reliability_gaps.head(50))


In [ ]:
# Hard-stop ledger: resolve every error before interpreting Q.
blocking = metadata_issues.loc[metadata_issues["severity"].astype(str).str.lower().eq("error")].copy()
save_table(blocking, "00_preflight", "blocking_metadata_issues")
print(f"Blocking metadata rows: {len(blocking):,}")
if len(blocking):
    display(blocking.head(50))
